# Phase 0: 50アーティストのデータ実現性検証

MusicBrainzでのMBID解決、ListenBrainzの聴取データ有無、Wikidataのジャンル等の被覆を検査します。同名誤結合は自動判定で終わらせず、最後に人手でMBIDを確認します。

## 実行前の注意

MusicBrainzに送るUser-Agentに連絡先が必要です。ターミナルで `export PROJECT_CONTACT=your-email@example.com` のように設定してからNotebookを起動してください。MusicBrainzへのリクエストは1秒以上間隔を置きます。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'validation_artists.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.feasibility import read_rows, run_validation

INPUT_PATH = PROJECT_ROOT / 'data' / 'validation_artists.csv'
REPORT_DIR = PROJECT_ROOT / 'reports'
CONTACT = os.environ.get('PROJECT_CONTACT', '')
if not CONTACT:
    raise RuntimeError('PROJECT_CONTACTに連絡可能なメールアドレスまたはURLを設定してください')
USER_AGENT = f'open-artist-discovery-feasibility/0.1 ({CONTACT})'

In [ ]:
artists = pd.DataFrame(read_rows(INPUT_PATH))
display(artists.groupby('category', dropna=False).size().rename('artists'))
display(artists.head())

## API取得とレポート生成

ListenBrainzのレスポンスに含まれるユーザー名は保存しません。このセルは約1分かかります。

In [ ]:
rows, summary = run_validation(INPUT_PATH, REPORT_DIR, USER_AGENT)
summary

In [ ]:
coverage = pd.DataFrame(rows)
columns = [
    'artist_name', 'category', 'resolution_status', 'resolved_mbid',
    'resolved_name', 'resolved_country', 'resolved_type',
    'resolved_disambiguation', 'listenbrainz_has_data',
    'total_listen_count', 'wikidata_item', 'wikidata_genre_count'
]
display(coverage[[column for column in columns if column in coverage.columns]])

## 人手確認

`reports/artist_coverage.csv` のMBIDと上位候補を確認し、`data/validation_artists.csv` の `manual_identity_status` を各行 `confirmed` または `incorrect` にします。50件が確認済みになるまで最終Go / No-Goは保留されます。

In [ ]:
review_columns = [
    'artist_name', 'disambiguation_hint', 'resolution_status',
    'resolved_mbid', 'resolved_name', 'resolved_disambiguation',
    'top_candidates', 'manual_identity_status'
]
display(coverage[review_columns])